# Fashion-MNIST: Linear Softmax Classifier

This note studies **multinomial logistic regression**, implemented as one affine layer followed by a softmax function, with no hidden layers and no regularization. The goal is to establish a mathematically consistent baseline for subsequent GD and SGD analysis.

The derivations distinguish derivatives with respect to **logits** from derivatives with respect to the **trainable parameters**.

## 1. Notation and dimensions

Vectors are column vectors in the mathematics. Samples are stored as rows in the data and logit matrices. Lowercase letters denote scalars or vectors, and uppercase letters denote matrices; indices make the distinction explicit.

| Symbol | Meaning | Dimension or range |
| --- | --- | --- |
| $n$ | Number of training samples | $60{,}000$ |
| $d$ | Number of features per image | $28\times28=784$ |
| $K$ | Number of classes | $10$ |
| $i$ | Sample index | $1,\ldots,n$ |
| $r,s$ | Feature indices | $1,\ldots,d$ |
| $j,\ell$ | Class indices | $1,\ldots,K$ |
| $T$ | Raw training images | $\{0,\ldots,255\}^{n\times28\times28}$ |
| $x_i$ | Flattened and normalized image | $[0,1]^d$ |
| $X$ | Matrix with row $i$ equal to $x_i^\top$ | $[0,1]^{n\times d}$ |
| $c_i$ | Class label in mathematical notation | $\{1,\ldots,K\}$ |
| $y_i$ | One-hot target, with $y_{i,j}=\mathbf{1}_{\{j=c_i\}}$ | $\{0,1\}^K$, $\mathbf{1}_K^\top y_i=1$ |
| $Y$ | Matrix with row $i$ equal to $y_i^\top$ | $\{0,1\}^{n\times K}$ |
| $W$ | Trainable weight matrix | $\mathbb{R}^{d\times K}$ |
| $b$ | Trainable bias vector | $\mathbb{R}^K$ |
| $z_i$ | Logit vector for sample $i$ | $\mathbb{R}^K$ |
| $Z$ | Matrix with row $i$ equal to $z_i^\top$ | $\mathbb{R}^{n\times K}$ |
| $p_i$ | Predicted class probabilities for sample $i$ | $(0,1)^K$, $\mathbf{1}_K^\top p_i=1$ |
| $P$ | Matrix with row $i$ equal to $p_i^\top$ | $(0,1)^{n\times K}$ |
| $\mathbf{1}_q$ | Vector of ones | $\mathbb{R}^q$ |

The mathematical class labels are one-based; the Python labels are zero-based, from `0` to `K - 1`. All norms of vectors below are Euclidean. Matrix parameter gradients use the Frobenius inner product, $\langle A,B\rangle_F=\operatorname{tr}(A^\top B)$.

## 2. Load and preprocess Fashion-MNIST

Fashion-MNIST supplies separate training and test sets, with $60{,}000$ and $10{,}000$ images, respectively. The objective below uses only the training set. The test set is reserved for evaluation.

For each image, flatten its $28\times28$ pixels and divide the raw integer intensities by $255$:

$$
x_i=\frac{\operatorname{flatten}(T_i)}{255}\in[0,1]^d.
$$

The code reads `dataset.data` directly. Dataset transforms such as `ToTensor()` are applied when retrieving samples through `dataset[i]`, not when directly accessing `.data`; therefore, the conversion and normalization are explicit here.

**Notebook usage:** Select a Python 3 kernel with `torch` and `torchvision` installed, then run the code cells in order. The first data-loading cell downloads Fashion-MNIST if needed. Code cells are intentionally saved without execution counts or outputs so they can be run in your own environment.


In [1]:
import torch
import torch.nn.functional as F

torch.manual_seed(42)


In [2]:
def load_fashion_mnist(root="data", download=False):
    """Read raw uint8 tensors and normalize once; transforms are unnecessary."""
    from torchvision import datasets

    training_data = datasets.FashionMNIST(
        root=root, train=True, download=download
    )
    test_data = datasets.FashionMNIST(
        root=root, train=False, download=download
    )
    X = training_data.data.flatten(start_dim=1).to(torch.float32) / 255.0
    labels = training_data.targets.to(torch.long)
    Y = F.one_hot(labels, num_classes=10).to(X.dtype)
    X_test = test_data.data.flatten(start_dim=1).to(torch.float32) / 255.0
    test_labels = test_data.targets.to(torch.long)
    return X, Y, labels, X_test, test_labels


In [3]:
# Set download=True for the first run if the dataset is not already present.
X, Y, labels, X_test, test_labels = load_fashion_mnist(
    root="data", download=True
)
n, d = X.shape
K = Y.shape[1]

assert X.shape == (60000, 784)
assert Y.shape == (60000, 10)
assert X_test.shape == (10000, 784)
assert torch.all(Y.sum(dim=1) == 1)


The one-hot matrix is convenient for the derivation. An implementation can instead keep integer labels to avoid storing it, but this note retains the original one-hot formulation.

## 3. Affine model and row-wise softmax

For fixed parameters $(W,b)$, define

$$
z_i=W^\top x_i+b,
\qquad
Z=XW+\mathbf{1}_n b^\top.
$$

In Python, `X @ W + b` broadcasts `b` over samples. The stored shape of `b` remains `(K,)`; broadcasting does not modify the tensor's shape.

The draft's initialization is retained: independently sample the entries of $W$ and $b$ uniformly from $[0,1)$. This initialization is a modeling choice, not an assumption required for the derivative formulas or convexity.


In [4]:
def initialize_parameters(d, K, *, dtype=torch.float32, device="gpu"):
    """Retain the draft's U[0, 1) initialization; seed the caller for repeats."""
    W = torch.rand(d, K, dtype=dtype, device=device)
    b = torch.rand(K, dtype=dtype, device=device)
    return W, b


In [5]:
W, b = initialize_parameters(d, K, dtype=X.dtype, device=X.device)
Z = X @ W + b
assert Z.shape == (n, K)


For one sample, define $\sigma:\mathbb{R}^K\to(0,1)^K$ by

$$
\sigma(z_i)_j=p_{i,j}
=\frac{\exp(z_{i,j})}{\sum_{\ell=1}^K\exp(z_{i,\ell})}.
$$

The corresponding matrix map $\Sigma:\mathbb{R}^{n\times K}\to(0,1)^{n\times K}$ applies $\sigma$ independently to each row, so $P=\Sigma(Z)$. Each row of $P$ sums to one.

**Flattening the dataset does not turn this into one softmax over $nK$ entries.** Every normalization denominator still includes only the $K$ classes of its own sample.

For numerical stability, introduce a separate shifted quantity

$$
m_i=\max_j z_{i,j},\qquad
\widetilde z_i=z_i-m_i\mathbf{1}_K.
$$

Softmax is invariant to this shift. The mathematical model continues to use the original logits $Z=XW+\mathbf{1}_n b^\top$; shifting is an evaluation technique.

## 4. Per-sample loss and scalar objective

With the target $y_i$ fixed, the per-sample negative log-likelihood is

$$
\ell_i:\mathbb{R}^K\to\mathbb{R},\qquad
\ell_i(z_i)
=-\sum_{j=1}^K y_{i,j}\log p_{i,j}
=-\log p_{i,c_i}.
$$

Using $\sum_j y_{i,j}=1$ gives the equivalent expression

$$ 
\begin{aligned}
\ell_i(z_i) &=-\sum_{j=1}^K y_{i,j}\log p_{i,j}\\
&=-\sum_{j=1}^K y_{i,j}\log\left(\frac{\exp(z_{i,j})}{\sum_{\ell=1}^K\exp(z_{i,\ell})}\right)\\
&=-\left(\sum_{j=1}^K y_{i,j} z_{i,j} - \underbrace{\sum_{j=1}^K y_{i,j}}_{=1} \log\sum_{j=1}^K\exp(z_{i,j}) \right)
\end{aligned}
$$

$$
\boxed{
\ell_i(z_i)=\log\left(\sum_{j=1}^K e^{z_{i,j}}\right)-y_i^\top z_i.
}
$$

For finite logits and $K\ge2$, $0<p_{i,c_i}<1$, so $\ell_i(z_i)>0$ in exact arithmetic. In particular, zero is a lower bound. Floating-point evaluation may round a sufficiently small loss to zero.

Distinguish three objects:

$$
\mathcal{L}(Z)=\begin{bmatrix}\ell_1(z_1)\\\vdots\\\ell_n(z_n)\end{bmatrix}
\in\mathbb{R}^n,
\qquad
\Phi(Z)=\frac1n\sum_{i=1}^n\ell_i(z_i),
$$

$$
\boxed{
f(W,b)=\Phi(XW+\mathbf{1}_n b^\top)
=\frac1n\sum_{i=1}^n f_i(W,b),
\qquad
f_i(W,b)=\ell_i(W^\top x_i+b).
}
$$

$\mathcal{L}$ is vector-valued, whereas $\Phi$ and $f$ are scalar-valued. Training minimizes $f$ over $(W,b)$; the logits are determined by the fixed data and these shared parameters.

### Stable implementation

Computing `torch.log(P)` after forming probabilities can produce `-inf` when a probability underflows to zero. Multiplying this by a zero one-hot entry can then produce `NaN`. Compute log-probabilities directly instead:

$$
\log p_{i,j}=\widetilde z_{i,j}
-\log\left(\sum_{\ell=1}^K e^{\widetilde z_{i,\ell}}\right).
$$


In [6]:
def log_softmax_rows(Z):
    """Stable row-wise log probabilities without taking log(softmax(Z))."""
    shifted = Z - Z.amax(dim=1, keepdim=True)
    return shifted - torch.logsumexp(shifted, dim=1, keepdim=True)


def forward(X, Y, W, b):
    """Y must contain one-hot rows (or valid probability distributions)."""
    Z = X @ W + b
    log_P = log_softmax_rows(Z)
    P = log_P.exp()
    losses = -(Y * log_P).sum(dim=1)
    return P, losses


def objective(X, Y, W, b):
    return forward(X, Y, W, b)[1].mean()


In [7]:
P, losses = forward(X, Y, W, b)
f_value = losses.mean()

assert losses.shape == (n,)
assert f_value.ndim == 0
torch.testing.assert_close(P.sum(dim=1), torch.ones_like(P[:, 0]))
torch.testing.assert_close(f_value, F.cross_entropy(X @ W + b, labels))


## 5. First-order derivatives

### 5.1 Softmax Jacobian for one sample

Temporarily omit the sample index and write $p=\sigma(z)$. For $j,\ell\in\{1,\ldots,K\}$,

$$
\frac{\partial p_j}{\partial z_\ell}
=\begin{cases}
p_j(1-p_j),&j=\ell,\\
-p_jp_\ell,&j\ne\ell.
\end{cases}
=p_j(\delta_{j\ell}-p_\ell).
$$

Thus,

$$
\boxed{J_\sigma(z)=\operatorname{Diag}(p)-pp^\top\in\mathbb{R}^{K\times K}.}
$$

Here $\delta_{j\ell}$ is the Kronecker delta.  

$$
\delta_{j\ell}
=\begin{cases}
1, j = \ell \\
0, j \neq \ell
\end{cases}
$$


Use **Jacobian** notation $J_\sigma$ (or $D\sigma$) for the derivative of this vector-valued function. Its entries are not all positive: the off-diagonal entries are negative for finite logits.

### 5.2 Loss gradient with respect to logits

For one sample, the chain rule gives

$$
\begin{aligned}
\frac{\partial\ell_i}{\partial z_{i,\ell}}
&=-\sum_{j=1}^K y_{i,j}\frac{1}{p_{i,j}}
\frac{\partial p_{i,j}}{\partial z_{i,\ell}}\\
&=-\sum_{j=1}^K y_{i,j}(\delta_{j\ell}-p_{i,\ell})\\
&=-\sum_{j=1}^K y_{i,j}\delta_{j\ell}+\sum_{j=1}^K y_{i,j}p_{i,\ell}\\
&=-y_{i,\ell}+p_{i,\ell}\underbrace{\sum_{j=1}^K y_{i,j}}_{=1}\\
&=p_{i,\ell}-y_{i,\ell}.
\end{aligned}
$$

Therefore,

$$
\boxed{\nabla_{z_i}\ell_i(z_i)=p_i-y_i\in\mathbb{R}^K,}
\qquad
\boxed{\nabla_Z\Phi(Z)=\frac1n(P-Y)\in\mathbb{R}^{n\times K}.}
$$

These gradients can have negative components even though the losses are nonnegative. A nonnegative loss does not imply a nonnegative gradient.

### 5.3 Gradients with respect to weights and bias

Since $z_{i,j}=\sum_{r=1}^d x_{i,r}w_{r,j}+b_j$,

$$
\frac{\partial f}{\partial w_{r,j}}
=\frac1n\sum_{i=1}^n x_{i,r}(p_{i,j}-y_{i,j}),
\qquad
\frac{\partial f}{\partial b_j}
=\frac1n\sum_{i=1}^n(p_{i,j}-y_{i,j}).
$$

In matrix form,

$$
\boxed{\nabla_W f(W,b)=\frac1n X^\top(P-Y)\in\mathbb{R}^{d\times K},}
$$

$$
\boxed{\nabla_b f(W,b)=\frac1n(P-Y)^\top\mathbf{1}_n\in\mathbb{R}^{K}.}
$$


In [8]:
def manual_gradient(X, Y, W, b):
    """Gradient of the MEAN loss; shapes are (d, K) and (K,)."""
    n = X.shape[0]
    P, _ = forward(X, Y, W, b)
    residual = P - Y
    grad_W = X.T @ residual / n
    grad_b = residual.mean(dim=0)
    return grad_W, grad_b


In [9]:
grad_W, grad_b = manual_gradient(X, Y, W, b)
assert grad_W.shape == W.shape  # (784, 10)
assert grad_b.shape == b.shape  # (10,)


The original expression `(X.T @ (P - Y)).mean(dim=0)` is incorrect: matrix multiplication has already summed over samples. Applying `.mean(dim=0)` afterward averages over the **feature** dimension, removes that dimension, and returns a vector of length $K$. Divide the matrix product by $n$ instead.

# Comparison of Assumptions in Stochastic Optimization Theory

We continue with the same mean cross-entropy objective and the same parameter gradients derived above. The aim is to check the assumptions mathematically before using a convergence theorem:

1. Prove global $L_i$-smoothness of each sample loss and obtain a smoothness constant for the mean objective.
2. Prove convexity, examine the effect of regularization, and distinguish convex from strongly convex convergence guarantees.
3. Compare stochastic-gradient assumptions using the terminology of [Khaled and Richtarik, *Better Theory for SGD in the Nonconvex World*, arXiv v3, Sections 3--4](https://arxiv.org/pdf/2002.03329v3).

The model here is convex, even though the referenced framework also covers nonconvex problems. The derivations below specialize the assumptions to our linear softmax model. Numerical checks verify the implementation; the proofs establish the statements for all parameter values.

## 6. Global smoothness of the sample and mean losses

### 6.1 Parameter coordinates and the Hessian

To discuss smoothness in **both weights and bias**, use the Euclidean parameter vector

$$
\theta=\begin{bmatrix}\operatorname{rvec}(W)\\b\end{bmatrix}
\in\mathbb{R}^{q},\qquad q=(d+1)K,
\qquad
\|\theta\|_2^2=\|W\|_F^2+\|b\|_2^2.
$$

Here $\operatorname{rvec}$ stacks matrix entries row by row, exactly as `W.reshape(-1)` does. Define

$$
a_i=\begin{bmatrix}x_i\\1\end{bmatrix},\qquad
A=[X\ \mathbf{1}_n],\qquad
B_i=a_i^\top\otimes I_K\in\mathbb{R}^{K\times q}.
$$

Then $z_i=B_i\theta$. From Section 5, differentiating $p_i-y_i$ with respect to $z_i$ gives

$$
\begin{aligned}
C_i:=\nabla_{z_i}^2\ell_i(z_i)
&=\nabla_{z_i}\left(p_i - y_i\right)\\
&=\operatorname{Diag}(p_i)-p_i p_i^\top.
\end{aligned}
$$

Since the logits are affine in $\theta$, their second derivatives vanish. Applying the chain rule twice yields

$$
\nabla_\theta f_i(\theta)=B_i^\top(p_i-y_i),
\qquad
\boxed{\nabla_\theta^2 f_i(\theta)
=B_i^\top C_i B_i=(a_i a_i^\top)\otimes C_i.}
$$

Consequently,

$$
\boxed{H(\theta):=\nabla_\theta^2 f(\theta)
=\frac1n\sum_{i=1}^n(a_i a_i^\top)\otimes C_i.}
$$

For augmented feature indices $r,s\in\{1,\ldots,d+1\}$, the entry with row $(r,j)$ and column $(s,\ell)$ is

$$
H_{(r,j),(s,\ell)}
=\frac1n\sum_{i=1}^n a_{i,r}a_{i,s}(C_i)_{j\ell}.
$$

This includes the weight--weight, weight--bias, and bias--bias blocks. The code constructs this matrix only for small verification problems: the Fashion-MNIST parameter Hessian would have $7{,}850^2$ entries.

In [10]:
def pack_parameters(W, b):
    return torch.cat((W.reshape(-1), b.reshape(-1)))


def unpack_parameters(theta, d, K):
    return theta[:d * K].reshape(d, K), theta[d * K:].reshape(K)


def logit_hessian_blocks(P):
    """C_i for each sample, shape (n, K, K), without the mean factor 1/n."""
    return torch.diag_embed(P) - P.unsqueeze(2) * P.unsqueeze(1)


def dense_parameter_hessian_tiny(X, W, b):
    """Parameter Hessian in row-major order; only for small problems."""
    n, d = X.shape
    K = W.shape[1]
    q = (d + 1) * K
    if q > 1000:
        raise ValueError("Use hessian_vector_product for more than 1000 parameters.")
    A = torch.cat((X, X.new_ones(n, 1)), dim=1)
    P = log_softmax_rows(X @ W + b).exp()
    C = logit_hessian_blocks(P)
    H4 = torch.einsum("ir,is,ijl->rjsl", A, A, C) / n
    return H4.reshape(q, q)

### 6.2 A uniform bound on the logit Hessian

For any $u\in\mathbb{R}^K$, let $\bar u=\sum_j p_{i,j}u_j$. Then

$$
\begin{aligned}
u^\top C_i u
&=\sum_jp_{i,j}u_j^2-\left(\sum_jp_{i,j}u_j\right)^2\\
&=\sum_jp_{i,j}(u_j-\bar u)^2\\
&=\sum_{j<\ell}p_{i,j}p_{i,\ell}(u_j-u_\ell)^2\ge0.
\end{aligned}
$$

To obtain an upper bound, use $(u_j-u_\ell)^2\le2(u_j^2+u_\ell^2)$:

$$
\begin{aligned}
u^\top C_i u
&\le2\sum_{j<\ell}p_{i,j}p_{i,\ell}(u_j^2+u_\ell^2)\\
&=2\sum_jp_{i,j}(1-p_{i,j})u_j^2\\
&\le\frac12\sum_j u_j^2.
\end{aligned}
$$

The last step uses $p(1-p)\le1/4$. Therefore,

$$
\boxed{0\preceq C_i\preceq\tfrac12 I_K\quad\text{for every finite }z_i.}
$$

The diagonal bound $1/4$ is **not** the spectral bound for the full softmax Hessian. For $K=2$ and $p=(1/2,1/2)$, $C_i$ has eigenvalues $0$ and $1/2$.

### 6.3 Smoothness in parameter space

Since $B_iB_i^\top=\|a_i\|_2^2I_K$, for every $v\in\mathbb{R}^q$,

$$
0\le v^\top\nabla^2 f_i(\theta)v
\le\frac12\|B_i v\|_2^2
\le\frac12\|a_i\|_2^2\|v\|_2^2.
$$

Thus one valid **global** smoothness constant is

$$
\boxed{L_i=\frac12\|a_i\|_2^2
=\frac12\left(\|x_i\|_2^2+1\right).}
$$

In particular, for any $\theta,\eta$, the fundamental theorem of calculus gives

$$
\begin{aligned}
\nabla f_i(\eta)-\nabla f_i(\theta)
&=\int_0^1\nabla^2f_i(\theta+t(\eta-\theta))(\eta-\theta)\,dt,\\
\|\nabla f_i(\eta)-\nabla f_i(\theta)\|_2
&\le L_i\|\eta-\theta\|_2.
\end{aligned}
$$

For the mean objective, summing the matrix inequalities gives

$$
0\preceq H(\theta)\preceq
\frac{1}{2n}(A^\top A)\otimes I_K.
$$

Hence the following are valid, increasingly conservative choices:

$$
\boxed{L_f=\frac{\|A\|_2^2}{2n}
\le L_{\mathrm{mean}}:=\frac1n\sum_i L_i
\le L_{\max}:=\max_iL_i.}
$$

Here $\|A\|_2$ is the matrix spectral norm, whereas $\|W\|_F$ is the Frobenius norm. These constants need not be the smallest possible constants. Because $x_i\in[0,1]^{784}$, we also have $L_i\le(784+1)/2=392.5$.

The default computation below uses $L_{\mathrm{mean}}$ and $L_{\max}$ and processes data in chunks. Computing $L_f$ additionally requires a $(d+1)\times(d+1)$ Gram matrix; we enable that only for the small example.

In [11]:
def smoothness_constants(X, *, spectral=False, chunk_size=4096):
    """Global bounds from the data, including the trainable bias coordinate."""
    n, d = X.shape
    L_i = torch.empty(n, dtype=torch.float64, device=X.device)
    gram = X.new_zeros((d + 1, d + 1), dtype=torch.float64) if spectral else None
    for start in range(0, n, chunk_size):
        Xi = X[start:start + chunk_size].to(torch.float64)
        L_i[start:start + len(Xi)] = (Xi.square().sum(dim=1) + 1) / 2
        if spectral:
            Ai = torch.cat((Xi, Xi.new_ones(len(Xi), 1)), dim=1)
            gram += Ai.T @ Ai
    result = {"per_sample": L_i, "mean": L_i.mean().item(), "max": L_i.max().item()}
    if spectral:
        result["spectral"] = torch.linalg.eigvalsh(gram)[-1].item() / (2 * n)
    return result


smoothness = smoothness_constants(X)
L_mean, L_max = smoothness["mean"], smoothness["max"]
assert L_mean <= L_max <= (d + 1) / 2 + 1e-10
print(f"Fashion-MNIST: L_mean = {L_mean:.6f}, L_max = {L_max:.6f}")
print(f"A sufficient full-gradient step size is 1/L_mean = {1/L_mean:.6g}")

Fashion-MNIST: L_mean = 81.426576, L_max = 262.724002
A sufficient full-gradient step size is 1/L_mean = 0.012281


### 6.4 Check the Hessian and the bounds numerically

Use a small, independent `float64` problem so that every Hessian entry can be compared with autograd. The reference loss is PyTorch's `F.cross_entropy`. A separate random generator preserves the previously initialized Fashion-MNIST parameters and random state.

We check the chain-rule formula, symmetry, nonnegative eigenvalues, the per-sample and mean spectral upper bounds, and one gradient Lipschitz inequality. Agreement at these points validates the code; it does not replace the global proof above.

In [12]:
check_rng = torch.Generator().manual_seed(123)
X_small = torch.randn(7, 3, generator=check_rng, dtype=torch.float64)
labels_small = torch.randint(4, (7,), generator=check_rng)
Y_small = F.one_hot(labels_small, num_classes=4).to(torch.float64)
W_small = 0.2 * torch.randn(3, 4, generator=check_rng, dtype=torch.float64)
b_small = 0.2 * torch.randn(4, generator=check_rng, dtype=torch.float64)
theta_small = pack_parameters(W_small, b_small)


def reference_loss(theta):
    W_ref, b_ref = unpack_parameters(theta, 3, 4)
    return F.cross_entropy(X_small @ W_ref + b_ref, labels_small)


H_small = dense_parameter_hessian_tiny(X_small, W_small, b_small)
H_autograd = torch.autograd.functional.hessian(reference_loss, theta_small)
torch.testing.assert_close(H_small, H_autograd, atol=1e-11, rtol=1e-10)
torch.testing.assert_close(H_small, H_small.T, atol=1e-12, rtol=0)

small_bounds = smoothness_constants(X_small, spectral=True)
eigenvalues = torch.linalg.eigvalsh(H_small)
assert eigenvalues[0].item() >= -1e-12
assert eigenvalues[-1].item() <= small_bounds["spectral"] + 1e-12
assert small_bounds["spectral"] <= small_bounds["mean"] + 1e-12
for i in range(len(X_small)):
    H_i = dense_parameter_hessian_tiny(X_small[i:i+1], W_small, b_small)
    assert torch.linalg.eigvalsh(H_i)[-1] <= small_bounds["per_sample"][i] + 1e-12

delta = torch.randn(theta_small.shape, generator=check_rng, dtype=torch.float64)
W_shifted, b_shifted = unpack_parameters(theta_small + delta, 3, 4)
g_small = pack_parameters(*manual_gradient(X_small, Y_small, W_small, b_small))
g_shifted = pack_parameters(*manual_gradient(X_small, Y_small, W_shifted, b_shifted))
assert torch.linalg.vector_norm(g_shifted - g_small) <= (
    small_bounds["spectral"] * torch.linalg.vector_norm(delta) + 1e-12
)
print(f"Hessian vs autograd: max error = {(H_small - H_autograd).abs().max().item():.3e}")
print(f"Local largest eigenvalue = {eigenvalues[-1].item():.6f}")
print(f"Global spectral bound   = {small_bounds['spectral']:.6f}")

Hessian vs autograd: max error = 2.776e-17
Local largest eigenvalue = 0.636194
Global spectral bound   = 1.081640


### 6.5 Hessian--vector products on the full dataset

For a parameter direction $v=(\operatorname{rvec}(V_W),v_b)$, define $R=XV_W+\mathbf{1}_n v_b^\top$. The class-space product is

$$
C_i r_i=p_i\odot\bigl(r_i-(p_i^\top r_i)\mathbf{1}_K\bigr).
$$

If these products form the rows of $Q$, then

$$
\boxed{(Hv)_W=\frac1nX^\top Q,\qquad (Hv)_b=\frac1nQ^\top\mathbf{1}_n.}
$$

This evaluates the exact Hessian action in real arithmetic using matrix multiplications, without allocating the dense parameter Hessian or all per-sample $K\times K$ blocks.

In [13]:
def hessian_vector_product(X, W, b, V_W, v_b):
    P = log_softmax_rows(X @ W + b).exp()
    R = X @ V_W + v_b
    Q = P * (R - (P * R).sum(dim=1, keepdim=True))
    return X.T @ Q / len(X), Q.mean(dim=0)


V_small, v_small = unpack_parameters(delta, 3, 4)
Hv_small = pack_parameters(*hessian_vector_product(
    X_small, W_small, b_small, V_small, v_small
))
torch.testing.assert_close(Hv_small, H_autograd @ delta, atol=1e-11, rtol=1e-10)

# A common shift of all class biases is a null direction, including on Fashion-MNIST.
Hv_W, Hv_b = hessian_vector_product(X, W, b, torch.zeros_like(W), torch.ones_like(b))
torch.testing.assert_close(Hv_W, torch.zeros_like(W), atol=1e-6, rtol=0)
torch.testing.assert_close(Hv_b, torch.zeros_like(b), atol=1e-6, rtol=0)
print("Hessian-vector product and common-bias-shift checks passed.")

Hessian-vector product and common-bias-shift checks passed.


## 7. Convexity and the effect of regularization

### 7.1 The original objective is convex, but not strongly convex

Section 6 proved that $\nabla^2f_i(\theta)\succeq0$ everywhere. Because these functions are twice continuously differentiable, each $f_i$ is convex, and their average $f$ is jointly convex in $(W,b)$.

However, for any $u\in\mathbb{R}^d$ and $c\in\mathbb{R}$, the transformation

$$
W\longmapsto W+u\mathbf{1}_K^\top,\qquad
b\longmapsto b+c\mathbf{1}_K
$$

adds the scalar $x_i^\top u+c$ to every class logit of sample $i$. Softmax probabilities, and therefore the loss, do not change. In particular,

$$
f(W,b+c\mathbf{1}_K)=f(W,b),\qquad
H(\theta)\begin{bmatrix}0\\\mathbf{1}_K\end{bmatrix}=0.
$$

A strongly convex function with modulus $\mu>0$ would require $v^\top H(\theta)v\ge\mu\|v\|_2^2$ for every $v$, which fails in this nonzero direction. Hence the original objective is neither strictly nor strongly convex on the full parameter space.

Also, $f\ge0$ does not guarantee a finite minimizer. On strictly separable data, scaling a separating parameter vector makes the loss approach zero without attaining zero at finite parameters. We do not assume that Fashion-MNIST is separable or that its unregularized minimizer exists. The convex objective-gap rates below explicitly assume a finite minimizer.

### 7.2 Which regularization makes the objective strongly convex?

There are two distinct issues with the draft penalty $\lambda\|W\|^2/(2n)$:

* Since $f$ is already an **average**, putting this penalty inside every $f_i$ leaves $\lambda\|W\|_F^2/(2n)$ after averaging. Its weight-space curvature is $\lambda/n$, rather than $\lambda$.
* Penalizing only $W$ preserves the flat direction $b\mapsto b+c\mathbf{1}_K$. It therefore does not produce joint strong convexity in $(W,b)$.

For a direct strongly convex comparison, define a new objective that penalizes **all** trainable coordinates:

$$
\boxed{F_\lambda(\theta)=f(\theta)+\frac\lambda2\|\theta\|_2^2
=f(W,b)+\frac\lambda2(\|W\|_F^2+\|b\|_2^2),\quad\lambda>0.}
$$

Its finite-sum components are $f_{i,\lambda}=f_i+\lambda\|\theta\|_2^2/2$, so $F_\lambda=n^{-1}\sum_i f_{i,\lambda}$ with no extra $1/n$ in the penalty. Direct differentiation gives

$$
\nabla F_\lambda=\nabla f+\lambda\theta,\qquad
\nabla^2F_\lambda=H+\lambda I_q.
$$

For any valid smoothness constant $L$ of $f$,

$$
\boxed{\lambda I_q\preceq\nabla^2F_\lambda
\preceq(L+\lambda)I_q.}
$$

Thus $F_\lambda$ is $\lambda$-strongly convex and $(L+\lambda)$-smooth. Since $F_\lambda(\theta)\ge\lambda\|\theta\|_2^2/2\to\infty$ as $\|\theta\|\to\infty$, continuity guarantees a minimizer; strong convexity makes it unique.

The optional `penalize_bias=False` below also implements the weight-only variant, but the strong-convexity statements apply to `penalize_bias=True`.

In [14]:
def regularized_objective(X, Y, W, b, lam=0.0, *, penalize_bias=True):
    if lam < 0:
        raise ValueError("lam must be nonnegative")
    penalty = W.square().sum()
    if penalize_bias:
        penalty = penalty + b.square().sum()
    return objective(X, Y, W, b) + 0.5 * lam * penalty


def regularized_gradient(X, Y, W, b, lam=0.0, *, penalize_bias=True):
    if lam < 0:
        raise ValueError("lam must be nonnegative")
    grad_W, grad_b = manual_gradient(X, Y, W, b)
    return grad_W + lam * W, grad_b + (lam * b if penalize_bias else 0)

In [15]:
lam_check = 0.1


def reference_regularized_loss(theta):
    return reference_loss(theta) + 0.5 * lam_check * theta.square().sum()


H_reg = torch.autograd.functional.hessian(reference_regularized_loss, theta_small)
identity = torch.eye(len(theta_small), dtype=torch.float64)
torch.testing.assert_close(H_reg, H_small + lam_check * identity, atol=1e-11, rtol=1e-10)
assert torch.linalg.eigvalsh(H_reg)[0] >= lam_check - 1e-12
g_reg = pack_parameters(*regularized_gradient(
    X_small, Y_small, W_small, b_small, lam_check
))
theta_check = theta_small.clone().requires_grad_(True)
g_reg_reference, = torch.autograd.grad(reference_regularized_loss(theta_check), theta_check)
torch.testing.assert_close(g_reg, g_reg_reference, atol=1e-11, rtol=1e-10)

penalty_mask = torch.cat((torch.ones(W_small.numel()), torch.zeros(b_small.numel()))).double()
H_weight_only = H_small + lam_check * torch.diag(penalty_mask)
bias_shift = pack_parameters(torch.zeros_like(W_small), torch.ones_like(b_small))
torch.testing.assert_close(H_weight_only @ bias_shift, torch.zeros_like(bias_shift), atol=1e-12, rtol=0)
print(f"Full regularization: minimum Hessian eigenvalue = {torch.linalg.eigvalsh(H_reg)[0].item():.6f}")
print("Weight-only regularization still has a zero-curvature bias direction.")

Full regularization: minimum Hessian eigenvalue = 0.100000
Weight-only regularization still has a zero-curvature bias direction.


## 8. Check the stochastic-gradient assumptions

### 8.1 Unbiased sampling and mini-batches

At iteration $t$, condition on the current parameters and all previous randomness. Draw $I$ uniformly from $\{1,\ldots,n\}$, independently of the past, and use

$$
g(\theta)=\nabla f_I(\theta),\qquad
\mathbb{E}[g(\theta)\mid\theta]
=\frac1n\sum_i\nabla f_i(\theta)=\nabla f(\theta).
$$

There is **no additional division by $n$** in a single-sample gradient. For a batch of $m$ independent draws **with replacement**, use

$$
g_m(\theta)=\frac1m\sum_{r=1}^m\nabla f_{I_r}(\theta).
$$

For $M_2(\theta)=n^{-1}\sum_i\|\nabla f_i(\theta)\|_2^2$, independence makes the cross terms factorize:

$$
\begin{aligned}
\mathbb{E}\|g_m\|_2^2
&=\frac{1}{m^2}\left(mM_2+m(m-1)\|\nabla f\|_2^2\right)\\
&=\boxed{\frac1mM_2+\left(1-\frac1m\right)\|\nabla f\|_2^2},\\
\mathbb{E}\|g_m-\nabla f\|_2^2
&=\boxed{\frac1m\left(M_2-\|\nabla f\|_2^2\right)}.
\end{aligned}
$$

For $F_\lambda$, add the deterministic vector $\lambda\theta$ **once** to the batch gradient. This preserves unbiasedness and leaves its variance unchanged at the same $\theta$:

$$
g_{m,\lambda}-\nabla F_\lambda=g_m-\nabla f.
$$

Sampling all $n$ indices with replacement is still random; it is not full GD. Random reshuffling and sampling without replacement have different conditional calculations and are not the scheme analyzed here.

In [16]:
def stochastic_gradient(X, Y, W, b, *, batch_size=1, lam=0.0, generator=None):
    """Uniform, independent sampling WITH replacement; penalty added once."""
    if batch_size < 1:
        raise ValueError("batch_size must be positive")
    indices = torch.randint(len(X), (batch_size,), device=X.device, generator=generator)
    return regularized_gradient(X[indices], Y[indices], W, b, lam)

### 8.2 Bounded second moments and bounded variance

Let $r_i=p_i-y_i$. Because $y_i$ is one-hot and $p_i$ is a probability vector,

$$
\|r_i\|_2^2=\|p_i\|_2^2+1-2p_{i,c_i}\le2.
$$

The sample gradient is the row-wise vectorization of $a_i r_i^\top$, so

$$
\boxed{\|\nabla f_i(\theta)\|_2^2
=\|a_i\|_2^2\|r_i\|_2^2
\le2\|a_i\|_2^2=4L_i.}
$$

It follows that, globally in $\theta$,

$$
M_2(\theta)\le G^2:=\frac2n\sum_i\|a_i\|_2^2=4L_{\mathrm{mean}},
\qquad
\mathbb{E}\|g_m-\nabla f\|_2^2\le\frac{G^2}{m}=: \sigma_m^2.
$$

The same variance bound applies to $g_{m,\lambda}$. Its **second moment** is not globally bounded by $G^2$, since the deterministic term $\lambda\theta$ grows with $\|\theta\|$. Keeping variance and second moment distinct matters for the strongly convex proof.

The following calculation uses the identity $\|a_i r_i^\top\|_F^2=\|a_i\|^2\|r_i\|^2$ without storing an $n\times q$ tensor of sample gradients. All moments are exact finite sums up to floating-point error, rather than Monte Carlo estimates.

In [17]:
def gradient_moments(X, Y, W, b, lam=0.0):
    """Uniform single-sample moments; use float64 for diagnostic comparisons."""
    if lam < 0:
        raise ValueError("lam must be nonnegative")
    P, losses = forward(X, Y, W, b)
    residual = P - Y
    a_norm_sq = X.square().sum(dim=1) + 1
    base_second = (a_norm_sq * residual.square().sum(dim=1)).mean()
    base_gradient = pack_parameters(X.T @ residual / len(X), residual.mean(dim=0))
    penalty_gradient = lam * pack_parameters(W, b)
    full_gradient = base_gradient + penalty_gradient
    second = (base_second + 2 * torch.dot(base_gradient, penalty_gradient)
              + penalty_gradient.square().sum())
    # The deterministic penalty cancels from variance. Roundoff can give a tiny negative value.
    variance = (base_second - base_gradient.square().sum()).clamp_min(0)
    return {
        "gradient": full_gradient,
        "gradient_norm_sq": full_gradient.square().sum().item(),
        "second_moment": second.item(),
        "variance": variance.item(),
        "loss": (losses.mean() + 0.5 * lam * pack_parameters(W, b).square().sum()).item(),
    }


# Independently enumerate sample gradients and every ordered pair on the tiny problem.
sample_gradients = torch.stack([
    pack_parameters(*manual_gradient(X_small[i:i+1], Y_small[i:i+1], W_small, b_small))
    for i in range(len(X_small))
])
moments_small = gradient_moments(X_small, Y_small, W_small, b_small)
torch.testing.assert_close(sample_gradients.mean(dim=0), moments_small["gradient"])
assert abs(sample_gradients.square().sum(dim=1).mean().item() - moments_small["second_moment"]) < 1e-12
pair_gradients = (sample_gradients[:, None, :] + sample_gradients[None, :, :]) / 2
pair_second = pair_gradients.square().sum(dim=2).mean().item()
pair_formula = (moments_small["second_moment"] + moments_small["gradient_norm_sq"]) / 2
assert abs(pair_second - pair_formula) < 1e-12
moments_reg = gradient_moments(X_small, Y_small, W_small, b_small, lam_check)
assert abs(moments_reg["variance"] - moments_small["variance"]) < 1e-12
print("Exact single-sample moments, batch-size-2 identity, and variance cancellation passed.")

Exact single-sample moments, batch-size-2 identity, and variance cancellation passed.


### 8.3 Expected smoothness and comparison with growth assumptions

We use the expected-smoothness form of [Khaled and Richtarik, Assumption 2](https://arxiv.org/pdf/2002.03329v3):

$$
\mathbb{E}\|g_m(\theta)\|_2^2
\le2A_{\mathrm{ES}}(f(\theta)-f_{\inf})
+B_{\mathrm{ES}}\|\nabla f(\theta)\|_2^2+C_{\mathrm{ES}},
\qquad f_{\inf}:=\inf_\theta f(\theta).
$$

The subscripts distinguish $A_{\mathrm{ES}}$ from our augmented data matrix $A$ and $C_{\mathrm{ES}}$ from the logit Hessians $C_i$.

**Derivation for this model.** Smoothness implies the descent inequality

$$
f_i(\eta)\le f_i(\theta)+\langle\nabla f_i(\theta),\eta-\theta\rangle
+\frac{L_i}{2}\|\eta-\theta\|_2^2.
$$

Insert $\eta=\theta-\nabla f_i(\theta)/L_i$ and use $f_i(\eta)\ge0$:

$$
0\le f_i(\theta)-\frac{\|\nabla f_i(\theta)\|_2^2}{2L_i},
\qquad
\boxed{\|\nabla f_i(\theta)\|_2^2\le2L_i f_i(\theta).}
$$

Average and apply the batch identity:

$$
\begin{aligned}
\mathbb{E}\|g_m\|_2^2
&\le\frac{2L_{\max}}{m}f(\theta)
+\left(1-\frac1m\right)\|\nabla f(\theta)\|_2^2\\
&=\frac{2L_{\max}}{m}(f(\theta)-f_{\inf})
+\left(1-\frac1m\right)\|\nabla f(\theta)\|_2^2
+\frac{2L_{\max}}{m}f_{\inf}.
\end{aligned}
$$

Thus valid constants are

$$
\boxed{A_{\mathrm{ES}}=\frac{L_{\max}}m,\quad
B_{\mathrm{ES}}=1-\frac1m,\quad
C_{\mathrm{ES}}=\frac{2L_{\max}f_{\inf}}m.}
$$

We have **not** proved $f_{\inf}=0$. A numerical check can evaluate the equivalent bound using $f(\theta)$ directly, without knowing the infimum. For full regularization, the same nonnegativity argument applies to $f_{i,\lambda}$ with constants $L_i+\lambda$, replacing $f$, $f_{\inf}$, and $L_{\max}$ accordingly.

The following comparison uses the condition names in Section 3 of the same paper; the conclusions follow from the model-specific calculations above.

| Condition | Mathematical statement | Conclusion here |
| --- | --- | --- |
| Bounded variance (BV) | $\mathbb{E}\|g_m-\nabla f\|^2\le\sigma_m^2$ | Holds with $\sigma_m^2=G^2/m$; also holds for full regularization. |
| Relaxed growth (RG) | $\mathbb{E}\|g_m\|^2\le\alpha\|\nabla f\|^2+\beta$ | Holds with $\alpha=1$, $\beta=G^2/m$, by unbiasedness and BV. The same statement holds with $F_\lambda$. |
| Expected strong growth (E-SG) | $\mathbb{E}\|g_m\|^2\le\alpha\|\nabla f\|^2$ | Not guaranteed: at a stationary point it would require every sampled gradient to vanish. |
| Maximal strong growth (M-SG) | $\|\nabla f_i\|^2\le\alpha\|\nabla f\|^2$ for every $i$ | Stronger than E-SG for single-sample uniform sampling; not guaranteed. |
| Expected smoothness (ES) | The three-term inequality above | Holds with the derived constants. |

For unregularized softmax with one-hot targets, each sample has a nonzero bias gradient at any finite $\theta$. Therefore E-SG fails at any finite stationary point of $f$, if such a point exists. This does not establish whether Fashion-MNIST has a finite stationary point. A simple counterexample with two identical inputs and opposite labels demonstrates why strong growth cannot follow merely from smoothness and convexity.

In [18]:
# Diagnostic on a clearly identified subset, with all 784 image features.
X_check = X[:512].to(torch.float64)
Y_check = Y[:512].to(torch.float64)
W_check, b_check = W.to(torch.float64), b.to(torch.float64)
check_bounds = smoothness_constants(X_check)
G_squared = 4 * check_bounds["mean"]
batch_check = 32
for lam_value in (0.0, 0.1):
    stats = gradient_moments(X_check, Y_check, W_check, b_check, lam_value)
    batch_second = (stats["second_moment"] / batch_check
                    + (1 - 1 / batch_check) * stats["gradient_norm_sq"])
    es_bound = (2 * (check_bounds["max"] + lam_value) * stats["loss"] / batch_check
                + (1 - 1 / batch_check) * stats["gradient_norm_sq"])
    assert stats["variance"] <= G_squared + 1e-9
    assert batch_second <= es_bound + 1e-9
    if lam_value == 0:
        assert stats["second_moment"] <= G_squared + 1e-9
    print(f"lambda={lam_value:g}: batch second moment={batch_second:.6f}, ES bound={es_bound:.6f}")

# Counterexample to E-SG and M-SG; this is not a claim about Fashion-MNIST's optimum.
X_conflict = torch.zeros(2, 1, dtype=torch.float64)
Y_conflict = torch.eye(2, dtype=torch.float64)
W_conflict = torch.zeros(1, 2, dtype=torch.float64)
b_conflict = torch.zeros(2, dtype=torch.float64)
conflict = gradient_moments(X_conflict, Y_conflict, W_conflict, b_conflict)
assert conflict["gradient_norm_sq"] == 0.0
assert abs(conflict["second_moment"] - 0.5) < 1e-12
print("Strong-growth counterexample: ||mean gradient||^2 = 0, E||sample gradient||^2 = 0.5")

lambda=0: batch second moment=15.680569, ES bound=70.291871
lambda=0.1: batch second moment=42.458497, ES bound=1869.351165
Strong-growth counterexample: ||mean gradient||^2 = 0, E||sample gradient||^2 = 0.5


## 9. Convergence consequences and matching update rules

Use $t$ for the iteration index and $N$ for the iteration horizon, so $K$ continues to mean the number of classes. Let $L$ be any of the proven smoothness bounds for $f$, and let $L_\lambda=L+\lambda$. All stochastic expectations below include the algorithm's sampling history.

### 9.1 Gradient descent

For an $L$-smooth objective, integration of the gradient along the segment from $\theta$ to $\eta$ gives the descent inequality used in Section 8. With $\theta_{t+1}=\theta_t-\gamma\nabla f(\theta_t)$,

$$
\begin{aligned}
f(\theta_{t+1})
&\le f(\theta_t)-\gamma\left(1-\frac{L\gamma}{2}\right)\|\nabla f(\theta_t)\|_2^2\\
&\le f(\theta_t)-\frac\gamma2\|\nabla f(\theta_t)\|_2^2,
\qquad 0<\gamma\le1/L.
\end{aligned}
$$

**Convex case.** Assume a finite minimizer $\theta^\star$ exists. Convexity gives $f(\theta_t)-f^\star\le\langle\nabla f(\theta_t),\theta_t-\theta^\star\rangle$. Combining this with the descent inequality and expanding the squared distance yields

$$
\begin{aligned}
f(\theta_{t+1})-f^\star
&\le\langle\nabla f(\theta_t),\theta_t-\theta^\star\rangle
-\frac\gamma2\|\nabla f(\theta_t)\|^2\\
&=\frac{\|\theta_t-\theta^\star\|^2-\|\theta_{t+1}-\theta^\star\|^2}{2\gamma}.
\end{aligned}
$$

Sum over $t=0,\ldots,N-1$ and use monotonicity of the loss:

$$
\boxed{f(\theta_N)-f^\star
\le\frac{\|\theta_0-\theta^\star\|^2}{2\gamma N}
=O(N^{-1}).}
$$

Without an attained minimum, descent still gives the stationarity bound
$\min_{0\le t<N}\|\nabla f(\theta_t)\|^2\le2(f(\theta_0)-f_{\inf})/(\gamma N)$.

**Strongly convex case.** Strong convexity of $F_\lambda$ implies

$$
F_\lambda^\star\ge F_\lambda(\theta)
+\langle\nabla F_\lambda(\theta),\theta^\star_\lambda-\theta\rangle
+\frac\lambda2\|\theta^\star_\lambda-\theta\|^2
\ge F_\lambda(\theta)-\frac{\|\nabla F_\lambda(\theta)\|^2}{2\lambda}.
$$

The second inequality minimizes the quadratic expression over the displacement. Therefore the Polyak--Lojasiewicz (PL) inequality holds:

$$
\|\nabla F_\lambda(\theta)\|^2
\ge2\lambda(F_\lambda(\theta)-F_\lambda^\star).
$$

Use this in the descent inequality, now with $0<\gamma\le1/L_\lambda$, to obtain

$$
\boxed{F_\lambda(\theta_N)-F_\lambda^\star
\le(1-\lambda\gamma)^N
\bigl(F_\lambda(\theta_0)-F_\lambda^\star\bigr).}
$$

This is a geometric, or linear, rate in the objective gap.

### 9.2 SGD for the convex objective

Assume a finite minimizer $\theta^\star$ exists. Let $\mathbb{E}_t$ condition on the history before drawing batch $t$. With a constant step $\gamma>0$ and $\theta_{t+1}=\theta_t-\gamma g_m(\theta_t)$,

$$
\begin{aligned}
\mathbb{E}_t\|\theta_{t+1}-\theta^\star\|^2
&=\|\theta_t-\theta^\star\|^2
-2\gamma\langle\nabla f(\theta_t),\theta_t-\theta^\star\rangle
+\gamma^2\mathbb{E}_t\|g_m(\theta_t)\|^2\\
&\le\|\theta_t-\theta^\star\|^2
-2\gamma(f(\theta_t)-f^\star)+\gamma^2G^2.
\end{aligned}
$$

The bound $\mathbb{E}_t\|g_m\|^2\le G^2$ follows from Jensen's inequality and the single-sample bound. After taking total expectations, summing, and using convexity at $\bar\theta_N=N^{-1}\sum_{t=0}^{N-1}\theta_t$,

$$
\boxed{\mathbb{E}[f(\bar\theta_N)]-f^\star
\le\frac{R_0^2}{2\gamma N}+\frac{\gamma G^2}{2},
\qquad R_0=\|\theta_0-\theta^\star\|.}
$$

For $R_0>0$, the theoretical choice $\gamma=R_0/(G\sqrt N)$ gives $R_0G/\sqrt N$. Since $R_0$ is unknown in practice, the code uses a stated constant times $N^{-1/2}$. This is an **averaged-iterate** guarantee; it does not assert monotone SGD losses or the same bound for the final iterate.

### 9.3 SGD for the strongly convex objective

For $F_\lambda$, use unbiasedness, $L_\lambda$-smoothness, and variance $\sigma_m^2=G^2/m$:

$$
\begin{aligned}
\mathbb{E}_tF_\lambda(\theta_{t+1})
&\le F_\lambda(\theta_t)-\gamma_t\|\nabla F_\lambda(\theta_t)\|^2
+\frac{L_\lambda\gamma_t^2}{2}
\left(\|\nabla F_\lambda(\theta_t)\|^2+\sigma_m^2\right)\\
&\le F_\lambda(\theta_t)-\frac{\gamma_t}{2}\|\nabla F_\lambda(\theta_t)\|^2
+\frac{L_\lambda\gamma_t^2\sigma_m^2}{2},
\qquad 0<\gamma_t\le1/L_\lambda.
\end{aligned}
$$

Let $\delta_t=\mathbb{E}[F_\lambda(\theta_t)-F_\lambda^\star]$. Applying PL gives

$$
\boxed{\delta_{t+1}\le(1-\lambda\gamma_t)\delta_t
+\frac{L_\lambda\gamma_t^2\sigma_m^2}{2}.}
$$

For a **constant** step, summing the geometric series yields

$$
\boxed{\delta_N\le(1-\lambda\gamma)^N\delta_0
+\frac{L_\lambda\gamma\sigma_m^2}{2\lambda}
\left(1-(1-\lambda\gamma)^N\right).}
$$

This guarantees a geometric transient toward a noise-dependent neighborhood, not convergence to the exact optimum. The upper bound alone does not prove that a nonzero limiting error actually occurs.

For a **decreasing** step, choose

$$
\gamma_t=\frac{2}{\lambda(t+t_0)},\qquad
t_0\ge\max\{2,2L_\lambda/\lambda\}.
$$

Write $D=\max\{t_0\delta_0,\,2L_\lambda\sigma_m^2/\lambda^2\}$. Induction gives $\delta_t\le D/(t+t_0)$: if $s=t+t_0\ge2$ and this holds at $t$, then

$$
\delta_{t+1}\le\left(1-\frac2s\right)\frac Ds
+\frac{2L_\lambda\sigma_m^2}{\lambda^2s^2}
\le\frac Ds-\frac D{s^2}\le\frac D{s+1}.
$$

Thus the last iterate satisfies $\boxed{\delta_N=O(N^{-1})}$. This bound is derived here from PL and bounded variance; we do not claim its constants are optimal.

If no unregularized minimizer is available, the same smoothness/variance argument without PL gives

$$
\min_{0\le t<N}\mathbb{E}\|\nabla f(\theta_t)\|^2
\le\frac{2(f(\theta_0)-f_{\inf})}{\gamma N}+L\gamma\sigma_m^2,
\qquad 0<\gamma\le1/L.
$$

Taking $\gamma$ proportional to $N^{-1/2}$ yields $O(N^{-1/2})$ for the **squared gradient norm**, rather than an objective-gap guarantee. These are different error measures.

### 9.4 Manual implementation and a small Fashion-MNIST run

The update rules below reuse `manual_gradient`; autograd is only a checker in earlier cells. `average_W` and `average_b` average the **pre-update** iterates $\theta_0,\ldots,\theta_{N-1}$ to match the convex SGD proof. Every run starts from copies of the same initial parameters and has its own sampling generator.

For a quick reproducible demonstration we use the first 512 training examples, all 784 features, and 80 updates. The smoothness constants are recomputed for this subset. This demonstrates the implementation, rather than measuring an asymptotic rate or reporting test accuracy. Pass the full `X, Y` to `run_optimizer` for a full-data run, using the corresponding full-data constants.

We log the objective being optimized and its full gradient norm. The regularized and unregularized objectives have different optimal values; their raw losses should not be interpreted as objective gaps or compared as evidence that one convergence rate is better. GD and SGD also use different numbers of sample gradients per update.

In [19]:
def run_optimizer(X, Y, W0, b0, *, method, steps, learning_rate,
                  lam=0.0, batch_size=32, seed=42, record_every=20):
    """Optimize f (lam=0) or F_lambda (both W and b penalized)."""
    if method not in {"gd", "sgd"}:
        raise ValueError("method must be 'gd' or 'sgd'")
    if steps < 1 or record_every < 1:
        raise ValueError("steps and record_every must be positive")
    W_run, b_run = W0.detach().clone(), b0.detach().clone()
    average_W, average_b = torch.zeros_like(W_run), torch.zeros_like(b_run)
    generator = torch.Generator(device=X.device).manual_seed(seed)
    history = []

    def record(step):
        value = regularized_objective(X, Y, W_run, b_run, lam).item()
        gradient = pack_parameters(*regularized_gradient(X, Y, W_run, b_run, lam))
        history.append((step, value, torch.linalg.vector_norm(gradient).item()))

    with torch.no_grad():
        record(0)
        for t in range(steps):
            average_W += W_run
            average_b += b_run
            rate = float(learning_rate(t) if callable(learning_rate) else learning_rate)
            if not 0 < rate < float("inf"):
                raise ValueError("learning_rate must be positive and finite")
            if method == "gd":
                grad_W, grad_b = regularized_gradient(X, Y, W_run, b_run, lam)
            else:
                grad_W, grad_b = stochastic_gradient(
                    X, Y, W_run, b_run, batch_size=batch_size,
                    lam=lam, generator=generator,
                )
            W_run -= rate * grad_W
            b_run -= rate * grad_b
            if (t + 1) % record_every == 0 or t + 1 == steps:
                record(t + 1)
    return {"W": W_run, "b": b_run, "average_W": average_W / steps,
            "average_b": average_b / steps, "history": history}

In [20]:
import math

steps_demo = 80
lam_demo = 0.1
L_demo = check_bounds["mean"]
L_regularized = L_demo + lam_demo
t0_demo = max(2, math.ceil(2 * L_regularized / lam_demo))
configurations = [
    ("GD convex", "gd", 0.0, 1 / L_demo),
    ("SGD convex", "sgd", 0.0, 1 / (L_demo * math.sqrt(steps_demo))),
    ("GD regularized", "gd", lam_demo, 1 / L_regularized),
    ("SGD regularized", "sgd", lam_demo, lambda t: 2 / (lam_demo * (t + t0_demo))),
]
runs = {}
print("First 512 training samples; objective values, not gaps to the optimum.")
print(f"{'run':<20} {'initial':>12} {'final':>12} {'averaged':>12} {'final |grad|':>14}")
for name, method, lam_value, schedule in configurations:
    result = run_optimizer(
        X_check, Y_check, W_check, b_check,
        method=method, steps=steps_demo, learning_rate=schedule,
        lam=lam_value, batch_size=32, seed=2026,
    )
    runs[name] = result
    history = result["history"]
    average_value = regularized_objective(
        X_check, Y_check, result["average_W"], result["average_b"], lam_value
    ).item()
    assert all(math.isfinite(value) and math.isfinite(norm) for _, value, norm in history)
    if method == "gd":
        assert all(after[1] <= before[1] + 1e-10 for before, after in zip(history, history[1:]))
    print(f"{name:<20} {history[0][1]:12.6f} {history[-1][1]:12.6f} "
          f"{average_value:12.6f} {history[-1][2]:14.6f}")

First 512 training samples; objective values, not gaps to the optimum.
run                       initial        final     averaged   final |grad|
GD convex                4.477305     2.044288     2.791673       1.013974
SGD convex               4.477305     3.729755     4.046397       2.180874
GD regularized         135.058858   109.030315   121.191514       4.743100
SGD regularized        135.058858   109.608058   121.428383       4.785454


### 9.5 What the proved rates compare

| Objective and method | Step-size choice | Quantity controlled | Guarantee |
| --- | --- | --- | --- |
| Convex $f$, GD | Constant $0<\gamma\le1/L$ | Last-iterate objective gap; assumes a finite minimizer | $O(N^{-1})$ |
| Convex $f$, SGD | Constant over a run, $\gamma\propto N^{-1/2}$ | Expected objective gap at $\bar\theta_N$; assumes a finite minimizer | $O(N^{-1/2})$ |
| Strongly convex $F_\lambda$, GD | Constant $0<\gamma\le1/L_\lambda$ | Last-iterate objective gap | $(1-\lambda\gamma)^N\delta_0$ |
| Strongly convex $F_\lambda$, SGD | Constant $0<\gamma\le1/L_\lambda$ | Expected last-iterate objective gap | Geometric transient plus an $O(\gamma\sigma_m^2)$ error bound, with $L_\lambda,\lambda$ fixed |
| Strongly convex $F_\lambda$, SGD | $\gamma_t=2/[\lambda(t+t_0)]$ with the stated $t_0$ | Expected last-iterate objective gap | $O(N^{-1})$ |

These results use the assumptions proved for this model. Smoothness alone does not imply strong convexity, a finite unregularized minimizer, or exact linear convergence of constant-step SGD. Expected smoothness and bounded variance both hold here; selecting a different valid bound changes the analysis, not the underlying objective or sampling algorithm.